# Parsiranje scenarija: *Fatalna nesanica* (*Light Sleeper*, 1992)

Izdvajanje iskaza Džona LeTura iz scenarija za film *Fatalna nesanica*. Ovo je jedan od četiri parsera urađena za master rad *Božji usamljeni ljudi: računarska analiza jezika protagonista u scenarijima Pola Šrejdera* (Univerzitet Singidunum, 2026). Svaki scenario u korpusu ima svoj parser, jer se četiri scenarija razlikuju po načinu na koji su zapisani ime lika, parentetika, dijalog i didaskalije. Postupak je opisan u potpoglavljima 5.2 i 5.3 rada.

**Izvor.** Scenario je preuzet sa sajta imsdb.com (tačna adresa navedena je u literaturi rada), u verziji za snimanje (*Shooting Draft*, 1992). Na sajtu je dostupan kao tekst, bez paginacije, pa je prekopiran u '.txt' fajl. Fajl nije deo repozitorijuma, jer je scenario zaštićen autorskim pravom. Za ponavljanje postupka tekst sa sajta treba prekopirati u 'Light_Sleeper.txt', tako da razmaci na početku redova ostanu sačuvani, jer parser na osnovu njih prepoznaje elemente scenarija.

**Izlaz.** 'letour_lines.csv', sa kolonama 'speaker', 'type', 'record_no' i 'line': 346 iskaza, od toga 341 dijalog i 5 naracije. Ovaj fajl je, zajedno sa izlazima ostala tri parsera, ulaz za notebook 'Analiza_scenarija.ipynb'.

**Pokretanje.** Redom, od prve ćelije. Od spoljnih paketa potreban je samo 'pandas'.

In [1]:
import re
import pandas as pd
from collections import Counter

## 1. Učitavanje scenarija

Scenario se čita red po red. Prazni redovi se preskaču, a za svaki preostali red beleže se redni broj u fajlu, broj razmaka na početku reda i sam tekst bez tih razmaka.

Broj razmaka ovde ima ulogu koju u scenarijima u PDF formatu ima x-koordinata. Scenario je zapisan sa doslednim uvlačenjem, pa svaki element scenarija počinje na svom, stalnom rastojanju od leve ivice. Zbog toga je ovaj parser najjednostavniji od četiri: tekst nema prelome i brojeve strana, oznake '(MORE)' i '(CONT'D)', revizione oznake ni druge ostatke PDF formata koje je u ostalim scenarijima trebalo ukloniti.

In [2]:
TXT_PATH = 'Light_Sleeper.txt'

with open(TXT_PATH, encoding='utf-8') as f:
    lines = f.read().splitlines()

all_rows = []
for i, l in enumerate(lines):
    if not l.strip():
        continue
    x0 = len(l) - len(l.lstrip(' ')) # broj razmaka na početku reda razlikuje tip sadržaja
    all_rows.append({'line_no': i + 1, 'x0': x0, 'text': l.strip()})

## 2. Ispravka i klasifikacija redova

Pri prenosu teksta sa sajta izgubljena su slova sa dijakritičkim znacima, na tri mesta u scenariju. Samo jedno od njih je u LeTurovoj replici ("Enchanté." je postalo "Enchant .")i ono se vraća ručno. Druga dva su u didaskaliji i u replici drugog lika, pa ne utiču na korpus.

Svaki red se zatim svrstava u jedan od pet tipova prema broju razmaka na početku
| Razmaci | Tip |
| --- | --- |
| 37 | ime lika |
| 30 | parentetika |
| 25 | dijalog |
| 15 | didaskalija |
| ostalo | ostalo |

Uz svaku vrednost dozvoljeno je odstupanje od dva razmaka, kako mala nepravilnost u uvlačenju ne bi promenila tip reda. U poslednju grupu ide sve što ne odgovara nijednom od četiri uvlačenja, i parser to tretira isto kao didaskaliju.

In [3]:
# Vraća Enchanté izgubljeno pri konverziji
ispravke = {'Enchant .': 'Enchanté.'}

# u zagradi je nominalno uvlačenje, opseg od +- 2 razmaka je tolerancija
def classify(x0):
    if 35 <= x0 <= 39: return 'speaker'  # ime lika (37)
    if 28 <= x0 <= 32: return 'paren'    # parentetika (30)
    if 23 <= x0 <= 27: return 'dialogue' # dijalog (25)
    if 13 <= x0 <= 17: return 'action'   # didaskalije (15)
    return 'other'

## 3. Parsiranje

Parser prolazi kroz revode redom i od njih sklapa iskaze. Ime lika otvara novi iskaz, redovi dijaloga dodaju se tekućem iskazu, a didaskalija ili bilo koji drugi red ga zatvara. Pri zatvaranju se redovi spajaju u jedan tekst, a višestruki razmaci svode na jedan. U ovom scenariju svaki blok dijaloga ima ime lika iznad sebe, pa se tim pravilom ne odbacuje nijedan red dijaloga.

Parentetika ne ulazi u tekst iskaza i ne prekida ga. Ako stoji usred replike, kao '(to TIS)' ili '(beat)', delovi pre i posle nje spajaju se u jedan iskaz, isto kao u ostala tri parsera.

Jedina parentetika koja se čita je oznaka naracije. U ovom scenariju ona nije sufiks uz ime lika, kao '(V.O.)' u ostala tri, nego zasebna parentetika '(voice over)' ispod imena. Kad je parser prepozna, tekući iskaz se označava kao naracija. Oznaka se proverava bez obzira na veličinu slova, jer je jednom napisana kao '(Voice over)'. Svako novo ime lika vraća oznaku na  dijalog, pa ona ne može preći na sledeći iskaz.

In [4]:
records = []
current, current_vo, buf = None, False, []

def flush():
    'Zatvori tekuću repliku i upiši je'
    global buf
    if current and buf:
        txt = re.sub(r'\s+', ' ', ' '.join(buf)).strip()
        if txt:
            records.append((current, current_vo, txt))
    buf = []
    
for r in all_rows:
    cls = classify(r['x0'])
    txt = ispravke.get(r['text'], r['text'])
    if cls == 'speaker':
        flush()
        current = txt
        current_vo = False
    elif cls == 'paren':
        # oznaka naracije je ovde parentetika ispod imena, a ne sufks;
        # lower() jer je jednom napisana kao (Voice over)
        # ostale parentetike se preskaču, pa se tekst pre i posle njih spaja u jedan iskaz
        if txt.lower() == '(voice over)':
            current_vo = True
    elif cls == 'dialogue':
        if current:
            buf.append(txt)
    else:
        # didaskalija zatvara iskaz; nastavak replike ovde uvek ima ponovo ime lika
        flush()
        current = None
flush()

### Kontrola: oblici parentetike

Pregled parentetika u scenariju po učestalosti. Svrha je provera da se naracija ne označava i na neki drugi način, npr. kao '(V.O.)'ili 
'(narration)'. Takvog oblika nema, pa je jedina oznaka naracije '(voice over)'.

U ispisu je taj oblik naveden četiri puta, a parser pronalazi pet iskaza naracije. Peti je napisan kao '(Voice over)', sa velikim početnim slovom, pa se pri brojanju vodi kao poseban oblik i, pošto se javlja samo jednom, ne ulazi u prvih trideset. Parser ga prepoznaje, jer oznaku poredi bez obzira na veličinu slova.

In [5]:
# kontrola: da li se naracija označava još nekim oblikom osim (voice over)
paren = [r['text'] for r in all_rows if classify(r['x0']) == 'paren']
print(Counter(paren).most_common(30))
print(Counter(p for p in paren if any(k in p.lower() for k in ('voice', 'v.o', 'narrat'))))                   

[('(out of shot)', 35), ('(beat)', 13), ('(to LETOUR)', 12), ('(nods)', 9), ('(to ROBERT)', 6), ('(to ANN)', 6), ('(voice over)', 4), ('(laughs)', 4), ('(on phone)', 4), ('("no")', 3), ('(corrects him)', 3), ('(overlap)', 2), ('(opens door)', 2), ('(stands)', 2), ('(to DRIVER)', 2), ('(smile)', 2), ('(he shrugs)', 2), ('(pause)', 2), ('(no answer)', 2), ('(after a moment)', 2), ('(to MARIANNE)', 2), ('(to TIS)', 2), ('(to THOMAS)', 2), ('(he does)', 2), ('(waves)', 1), ('(about ROBERT)', 1), ('(re-emerges)', 1), ('(LETOUR shrugs)', 1), ('(shrugs)', 1), ('(passes bag)', 1)]
Counter({'(voice over)': 4, '(Voice over)': 1, "(Marianne's voice)": 1})


## 4. Izdvajanje LeTurovih iskaza i čuvanje

Svi iskazi se najpre smeštaju u tabelu i dobijaju redni broj u celom scenariju ('record_no'). Broj se dodeljuje pre izdvajanja protagoniste, pa obuhvata replike svih likova, dok didaskalije broj ne dobijaju. Zato dva LeTurova iskaza sa uzastopnim brojevima nije razdvojila replika drugog lika, nego najviše didaskalija. Na tome počiva spajanje iskaza naracije u blokove u glavnom notebooku. Kod LeTura nijedan par iskaza naracije nije susedan, pa je svaki od pet iskaza zaseban blok.

Pre izdvajanja se rade dve kontrole. Prva traži oznaku 'V.O' u imenima likova. Ista provera postoji u sva četiri parsera, a ovde je ispis prazan, jer oznaka naracije u ovom scenariju nije deo imena. Druga je spisak likova po broju replika. Iz njega se vidi da su sve LeTurove replike označene istim imenom, 'LETOUR', iako ga didaskalije ponekad zovu Džon, pa ime nije potrebno ujednačavati, kao u ostalim parserima.

Mali broj iskaza naracije nije posledica parsiranja, nego same verzije scenarija. Scenario na dva mesta navodi da se naracija nastavlja, ali taj tekst nije ispisan. Ovo se razmatra u delu rada o ograničenjima.


In [6]:
df = pd.DataFrame(records, columns=['speaker_raw', 'is_vo', 'line'])
df['record_no'] = range(1, len(df) + 1) # redni broj u celom scenariju, potreban zbog kasnijeg spajanja blokova
df['speaker'] = df['speaker_raw']

print(sorted({s for s in df['speaker_raw'].unique() if 'V.O' in s.upper()}))
print(df['speaker_raw'].value_counts().head(20))

letour = df[df['speaker'] == 'LETOUR'].copy().reset_index(drop=True)
letour['type'] = letour['is_vo'].apply(lambda v: 'voice_over' if v else 'dialogue')
letour = letour[['speaker', 'type', 'record_no', 'line']]

print('LeTourovih replika:' , len(letour))
print(letour['type'].value_counts())

letour.to_csv('letour_lines.csv', index=False)

[]
speaker_raw
LETOUR                    346
ANN                       112
ROBERT                     65
MARIANNE                   62
TERESA                     32
TIS                        28
RANDI                      26
EDDIE                      16
MANUEL                     16
GUIDONE                    14
JEALOUS                    11
HOTEL SWITCHBOARD           7
YOUNG HASID                 5
MEDICAL VOICE               4
CARLOS                      4
MID-TWENTIESH MAN           2
FIRST STUDENT               2
SECOND STUDENT              2
ST. LUKE'S SWITCHBOARD      2
THOMAS                      2
Name: count, dtype: int64
LeTourovih replika: 346
type
dialogue      341
voice_over      5
Name: count, dtype: int64
